### 读数据

In [2]:
import pandas as pd

# 1. 读取数据
df = pd.read_csv("ruc_Class25Q2_train_price.csv")
df_test = pd.read_csv("ruc_Class25Q2_test_price.csv")

C:\Users\heart\AppData\Local\Temp\ipykernel_24152\2208309609.py:4: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("ruc_Class25Q2_train_price.csv")
C:\Users\heart\AppData\Local\Temp\ipykernel_24152\2208309609.py:5: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv("ruc_Class25Q2_test_price.csv")


In [3]:
df.drop(columns=["抵押信息"], inplace=True)
df_test.drop(columns=["抵押信息"], inplace=True)

#### 正则提取面积

In [4]:
import pandas as pd
import numpy as np
import re

# 提取数字的函数
def extract_numeric(value):
    if pd.isnull(value):
        return np.nan
    # 提取所有数字（包括小数）
    nums = re.findall(r"\d+\.?\d*", str(value))
    if not nums:
        return np.nan
    nums = list(map(float, nums))
    # 若为区间（如2.61-2.63），取平均值
    return np.mean(nums)

# 需要处理的列
cols_to_clean = ["建筑面积", "套内面积", "燃气费", "供热费"]

# 检查哪些列在 DataFrame 中存在
cols_to_clean = [col for col in cols_to_clean if col in df.columns]

# 对每列应用提取函数
for col in cols_to_clean:
    df[col] = df[col].apply(extract_numeric)

# 查看结果前几行
print(df[cols_to_clean].head())


     建筑面积    套内面积   燃气费   供热费
0   52.30     NaN  2.61  30.0
1  127.44  123.70  2.61   NaN
2  118.02  101.95  2.61  30.0
3  293.23  293.23  2.62   NaN
4   39.85   29.94  2.62  37.5


In [5]:
import pandas as pd
import numpy as np
import re

# 提取数字的函数
def extract_numeric(value):
    if pd.isnull(value):
        return np.nan
    # 提取所有数字（包括小数）
    nums = re.findall(r"\d+\.?\d*", str(value))
    if not nums:
        return np.nan
    nums = list(map(float, nums))
    # 若为区间（如2.61-2.63），取平均值
    return np.mean(nums)

# 需要处理的列
cols_to_clean = ["建筑面积", "套内面积", "燃气费", "供热费"]

# 检查哪些列在 DataFrame 中存在
cols_to_clean = [col for col in cols_to_clean if col in df_test.columns]

# 对每列应用提取函数
for col in cols_to_clean:
    df_test[col] = df_test[col].apply(extract_numeric)

# 查看结果前几行
print(df_test[cols_to_clean].head())


     建筑面积    套内面积   燃气费   供热费
0  282.02     NaN  2.61   NaN
1   88.42   71.78  2.61  30.0
2  175.52  139.86  2.61  30.0
3  106.13     NaN  2.61  30.0
4  116.80     NaN  2.62  27.0


### 处理环线

#### 给环线填空值KNN

In [6]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors

def impute_ring_by_knn(df,
                       city_col='城市', region_col='区域', plate_col='板块',
                       lon_col='lon', lat_col='lat',
                       ring_col='环线',
                       k=8,
                       weight_by_dist=True,
                       eps=1e-6,
                       inplace=False,
                       random_state=0):
    df_use = df if inplace else df.copy()

    # 1) 分出已知与缺失的索引
    mask_known = df_use[ring_col].notna()
    mask_missing = ~mask_known

    n_known = mask_known.sum()
    n_missing = mask_missing.sum()
    print(f"总行数 {len(df_use)}，已知 {n_known} 行，缺失 {n_missing} 行（将被填补）")

    if n_missing == 0:
        print("没有缺失，直接返回。")
        return df_use

    # 2) 构建特征矩阵：one-hot(city, region, plate) + scaled(lon, lat)
    cat_cols = [city_col, region_col, plate_col]
    num_cols = [lon_col, lat_col]

    # 检查所需列是否存在
    for c in cat_cols + num_cols + [ring_col]:
        if c not in df_use.columns:
            raise KeyError(f"缺少列: {c}")

    # 用于编码的行（我们用所有行的类别信息来 fit encoder，以避免 test-only 类问题）
    cat_matrix = df_use[cat_cols].astype(object).fillna(-9999)  # 把缺失临时标记一个特殊值（避免报错）
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    ohe.fit(cat_matrix)


    def build_feature_matrix(df_rows):
        cat_part = ohe.transform(df_rows[cat_cols].astype(object).fillna(-9999))
        # 数值列：若有缺失，填 0（或均值），这里用 0 并后续标准化
        num_part = df_rows[num_cols].astype(float).fillna(0.0).values
        # 标准化数值列
        return cat_part, num_part

    if n_known >= 2:
        scaler = StandardScaler()
        scaler.fit(df_use.loc[mask_known, num_cols].astype(float).fillna(0.0).values)
    else:
        scaler = StandardScaler()
        scaler.fit(df_use[num_cols].astype(float).fillna(0.0).values)

    cat_known, num_known = build_feature_matrix(df_use.loc[mask_known])
    cat_miss, num_miss = build_feature_matrix(df_use.loc[mask_missing])

    num_known_s = scaler.transform(num_known)
    num_miss_s = scaler.transform(num_miss)

    X_known = np.hstack([cat_known, num_known_s])
    X_miss = np.hstack([cat_miss, num_miss_s])

    # 3) 准备训练标签（分类字符串）
    y_known = df_use.loc[mask_known, ring_col].astype(object).values  # strings like '二至三环' etc.

    # 4) fit nearest neighbors on known rows
    if X_known.shape[0] == 0:
        print("训练集中没有已知环线，无法填充。")
        return df_use

    nn = NearestNeighbors(n_neighbors=min(k, X_known.shape[0]), metric='euclidean')
    nn.fit(X_known)

    # 5) 对每一个缺失样本查找 neighbors 并做投票
    dists, inds = nn.kneighbors(X_miss, return_distance=True)
    # dists shape (n_missing, k'), inds shape (n_missing, k')

    filled = []
    classes = np.unique(y_known)  # 所有已见类别
    for i_row in range(X_miss.shape[0]):
        neigh_idx = inds[i_row]  # indices into known set
        neigh_dists = dists[i_row]
        neigh_labels = y_known[neigh_idx]


        if weight_by_dist:
            weights = 1.0 / (neigh_dists + eps)
        else:
            weights = np.ones_like(neigh_dists)

        vote_dict = {}
        for lbl, w in zip(neigh_labels, weights):
            vote_dict[lbl] = vote_dict.get(lbl, 0.0) + float(w)

        max_w = max(vote_dict.values())
        candidates = [lbl for lbl, w in vote_dict.items() if abs(w - max_w) < 1e-12]
        if len(candidates) == 1:
            chosen = candidates[0]
        else:

            for idx_n, lbl in zip(neigh_idx, neigh_labels):
                if lbl in candidates:
                    chosen = lbl
                    break

        filled.append(chosen)

    miss_indices = df_use.loc[mask_missing].index
    for idx, val in zip(miss_indices, filled):
        df_use.at[idx, ring_col] = val

    print(f"填补完成: 已为 {len(filled)} 个缺失值赋予环线类别。若仍有缺失，表示训练集中无样本可参照。")
    return df_use

df = impute_ring_by_knn(df, city_col='城市', region_col='区域', plate_col='板块',
                                lon_col='lon', lat_col='lat', ring_col='环线', k=8)
df.head()

总行数 103871，已知 40419 行，缺失 63452 行（将被填补）


C:\Users\heart\AppData\Local\Temp\ipykernel_24152\3867870567.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cat_matrix = df_use[cat_cols].astype(object).fillna(-9999)  # 把缺失临时标记一个特殊值（避免报错）
C:\Users\heart\AppData\Local\Temp\ipykernel_24152\3867870567.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cat_part = ohe.transform(df_rows[cat_cols].astype(object).fillna(-9999))
C:\Users\heart\AppData\Local\Temp\ipykernel_24152\3867870567.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a f

填补完成: 已为 63452 个缺失值赋予环线类别。若仍有缺失，表示训练集中无样本可参照。


,城市,区域,板块,环线,Price,房屋户型,所在楼层,建筑面积,套内面积,房屋朝向,...,供水,供暖,供电,燃气费,供热费,停车位,停车费用,coord_x,coord_y,客户反馈
0,0,109.0,150.0,二至三环,6.194049e+06,2室1厅1厨1卫,中楼层 (共5层),52.30,NaN,南 北,...,民水,集中供暖,民电,2.61,30.0,300.0,暂无,117.424278,40.975752,听说，设施老旧，停车费高
1,0,65.0,299.0,五至六环,4.354153e+06,3室1厅1厨1卫,顶层 (共6层),127.44,123.70,南 北,...,商水/民水,自采暖,商电/民电,2.61,NaN,1550.0,150,117.389228,41.091295,整体印象，网速快，面积适中
2,0,62.0,911.0,五至六环,3.321992e+06,3室2厅1厨2卫,低楼层 (共6层),118.02,101.95,东南,...,商水/民水,集中供暖/自采暖,商电/民电,2.61,30.0,324.0,150,117.200934,40.747919,地段一般，停车划线清晰，说白了，居住体验佳
3,0,123.0,1102.0,六环外,7.895656e+06,6室3厅1厨3卫,底层 (共2层),293.23,293.23,东 南 西 北,...,民水,自采暖,民电,2.62,NaN,500.0,暂无,117.767308,41.228803,暖气好，平均水准，厨房设备新，不过话说，气味中性
4,0,81.0,295.0,三至四环,1.902960e+06,1房间1卫,中楼层 (共10层),39.85,29.94,南,...,商水/民水,集中供暖/自采暖,商电/民电,2.62,37.5,1800.0,1200,117.334530,40.952530,可以说，通风一般，空气清新，换个角度看，气味刺鼻


##### 测试集

In [7]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors

def impute_ring_by_knn(df,
                       city_col='城市', region_col='区域', plate_col='板块',
                       lon_col='lon', lat_col='lat',
                       ring_col='环线',
                       k=8,
                       weight_by_dist=True,
                       eps=1e-6,
                       inplace=False,
                       random_state=0):
    df_use = df if inplace else df.copy()

    # 1) 分出已知与缺失的索引
    mask_known = df_use[ring_col].notna()
    mask_missing = ~mask_known

    n_known = mask_known.sum()
    n_missing = mask_missing.sum()
    print(f"总行数 {len(df_use)}，已知 {n_known} 行，缺失 {n_missing} 行（将被填补）")

    if n_missing == 0:
        print("没有缺失，直接返回。")
        return df_use

    # 2) 构建特征矩阵：one-hot(city, region, plate) + scaled(lon, lat)
    cat_cols = [city_col, region_col, plate_col]
    num_cols = [lon_col, lat_col]

    # 检查所需列是否存在
    for c in cat_cols + num_cols + [ring_col]:
        if c not in df_use.columns:
            raise KeyError(f"缺少列: {c}")

    # 用于编码的行（我们用所有行的类别信息来 fit encoder，以避免 test-only 类问题）
    cat_matrix = df_use[cat_cols].astype(object).fillna(-9999)  # 把缺失临时标记一个特殊值（避免报错）
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    ohe.fit(cat_matrix)


    def build_feature_matrix(df_rows):
        cat_part = ohe.transform(df_rows[cat_cols].astype(object).fillna(-9999))
        # 数值列：若有缺失，填 0（或均值），这里用 0 并后续标准化
        num_part = df_rows[num_cols].astype(float).fillna(0.0).values
        # 标准化数值列
        return cat_part, num_part

    if n_known >= 2:
        scaler = StandardScaler()
        scaler.fit(df_use.loc[mask_known, num_cols].astype(float).fillna(0.0).values)
    else:
        scaler = StandardScaler()
        scaler.fit(df_use[num_cols].astype(float).fillna(0.0).values)

    cat_known, num_known = build_feature_matrix(df_use.loc[mask_known])
    cat_miss, num_miss = build_feature_matrix(df_use.loc[mask_missing])

    num_known_s = scaler.transform(num_known)
    num_miss_s = scaler.transform(num_miss)

    X_known = np.hstack([cat_known, num_known_s])
    X_miss = np.hstack([cat_miss, num_miss_s])

    # 3) 准备训练标签（分类字符串）
    y_known = df_use.loc[mask_known, ring_col].astype(object).values  # strings like '二至三环' etc.

    # 4) fit nearest neighbors on known rows
    if X_known.shape[0] == 0:
        print("训练集中没有已知环线，无法填充。")
        return df_use

    nn = NearestNeighbors(n_neighbors=min(k, X_known.shape[0]), metric='euclidean')
    nn.fit(X_known)

    # 5) 对每一个缺失样本查找 neighbors 并做投票
    dists, inds = nn.kneighbors(X_miss, return_distance=True)
    # dists shape (n_missing, k'), inds shape (n_missing, k')

    filled = []
    classes = np.unique(y_known)  # 所有已见类别
    for i_row in range(X_miss.shape[0]):
        neigh_idx = inds[i_row]  # indices into known set
        neigh_dists = dists[i_row]
        neigh_labels = y_known[neigh_idx]


        if weight_by_dist:
            weights = 1.0 / (neigh_dists + eps)
        else:
            weights = np.ones_like(neigh_dists)

        vote_dict = {}
        for lbl, w in zip(neigh_labels, weights):
            vote_dict[lbl] = vote_dict.get(lbl, 0.0) + float(w)

        max_w = max(vote_dict.values())
        candidates = [lbl for lbl, w in vote_dict.items() if abs(w - max_w) < 1e-12]
        if len(candidates) == 1:
            chosen = candidates[0]
        else:

            for idx_n, lbl in zip(neigh_idx, neigh_labels):
                if lbl in candidates:
                    chosen = lbl
                    break

        filled.append(chosen)

    miss_indices = df_use.loc[mask_missing].index
    for idx, val in zip(miss_indices, filled):
        df_use.at[idx, ring_col] = val

    print(f"填补完成: 已为 {len(filled)} 个缺失值赋予环线类别。若仍有缺失，表示训练集中无样本可参照。")
    return df_use

df_test = impute_ring_by_knn(df_test, city_col='城市', region_col='区域', plate_col='板块',
                                lon_col='lon', lat_col='lat', ring_col='环线', k=8)
df_test.head()

总行数 34017，已知 15677 行，缺失 18340 行（将被填补）


C:\Users\heart\AppData\Local\Temp\ipykernel_24152\124117638.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cat_matrix = df_use[cat_cols].astype(object).fillna(-9999)  # 把缺失临时标记一个特殊值（避免报错）
C:\Users\heart\AppData\Local\Temp\ipykernel_24152\124117638.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cat_part = ohe.transform(df_rows[cat_cols].astype(object).fillna(-9999))
C:\Users\heart\AppData\Local\Temp\ipykernel_24152\124117638.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a futu

填补完成: 已为 18340 个缺失值赋予环线类别。若仍有缺失，表示训练集中无样本可参照。


,ID,城市,区域,板块,环线,房屋户型,所在楼层,建筑面积,套内面积,房屋朝向,...,供水,供暖,供电,燃气费,供热费,停车位,停车费用,coord_x,coord_y,客户反馈
0,1000000,0,109.0,367.0,二至三环,3室2厅1厨2卫,中楼层 (共23层),282.02,NaN,南 北,...,民水,自采暖,民电,2.61,NaN,280.0,地上150元/月/位，地下2元/时/位，地下固定车位450元/月/位,117.389491,40.901030,卫生差，阳光充足
1,1000001,0,28.0,606.0,五至六环,2室1厅1厨1卫,中楼层 (共17层),88.42,71.78,南 北,...,商水/民水,集中供暖,商电/民电,2.61,30.0,1340.0,地上150元，地上机械180，地下300,117.376625,40.767478,卫生间整洁，通风死角多
2,1000002,0,123.0,1110.0,五至六环,3室1厅1厨2卫,高楼层 (共12层),175.52,139.86,西北,...,民水,集中供暖,民电,2.61,30.0,300.0,150,117.631276,41.063635,反过来看，室内采光均衡，听说，监控覆盖
3,1000003,0,65.0,555.0,六环外,2室1厅1厨1卫,中楼层 (共5层),106.13,NaN,南 北,...,民水,集中供暖/自采暖,民电,2.61,30.0,500.0,暂无,117.186216,41.163738,话又说回来，电力稳定，总体趋势上，网速快，门窗紧实
4,1000004,0,109.0,990.0,二环内,3室2厅1厨2卫,顶层 (共5层),116.80,NaN,南 北,...,商水/民水,集中供暖,民电,2.62,27.0,80.0,150,117.400114,40.959679,总体状况一般，门窗紧实，个人觉得，物业服务好


In [8]:
import pandas as pd

# 1. 检查空值（缺失值）数量
null_count = df['环线'].isnull().sum()
print(f"环线列的空值数量：{null_count}")

# 2. 检查唯一值（unique）数量
unique_count = df['环线'].nunique()
print(f"环线列的唯一值数量：{unique_count}")

# 3. 查看所有唯一值（中文描述会完整显示）
unique_values = df['环线'].unique()
print("环线列的所有唯一值：")
for value in unique_values:
    print(f"- {value}")

环线列的空值数量：0
环线列的唯一值数量：11
环线列的所有唯一值：
- 二至三环
- 五至六环
- 六环外
- 三至四环
- 四至五环
- 二环内
- 内环内
- 内环至外环
- 外环外
- 内环至中环
- 中环至外环


#### loc_ring_二至三环命名，方便后续操作

In [9]:
categories = [
    #"二至三环"
    "五至六环",
    "六环外",
    "三至四环",
    "四至五环",
    "二环内",
    "内环内",
    "内环至外环",
    "外环外",
    "内环至中环",
    "中环至外环"
]
cats_clean = [c.strip() for c in categories]

# 生成目标列名
dummy_cols = [f"loc_ring_{c}" for c in cats_clean]

# 初始化 dummies DataFrame（float，以便放 NaN）
dummies = pd.DataFrame(0.0, index=df.index, columns=dummy_cols)

# 原始列做 strip 处理以容忍前后空格
orig = df['环线'].astype(object).map(lambda x: x.strip() if pd.notna(x) else x)

# 对于原始为空的行，把所有虚拟列设为 NaN
mask_na = orig.isna()
dummies.loc[mask_na, :] = np.nan

# 对非空行，按类别赋值（匹配 cats_clean）
mask_non_na = ~mask_na
for idx, val in orig[mask_non_na].items():
    if val in cats_clean:
        colname = f"loc_ring_{val}"
        dummies.at[idx, colname] = 1.0
    else:
        # 如果遇到未列入 categories 的值，保持该行所有 dummy 为 0（并记录以便检查）
        pass

# 把这些新列拼回原 df（如需单独保存也可）
df = pd.concat([df, dummies], axis=1)

# 打印检查信息
print("创建的虚拟变量列：", dummy_cols)
# 检查原始中未包含在 categories 内的值
orig_vals = set(df['环线'].dropna().astype(str).map(str.strip).unique())
missing_vals = sorted(list(orig_vals - set(cats_clean)))
if missing_vals:
    print("注意：以下环线值出现在 df 中但未列入 categories（示例）:", missing_vals[:30])
else:
    print("所有 df 中出现的非空环线值都已包含在 categories 中。")

# 示例查看
print("\n示例（前8行）：")
print(pd.concat([df['环线'].head(8), dummies.head(8)], axis=1))


创建的虚拟变量列： ['loc_ring_五至六环', 'loc_ring_六环外', 'loc_ring_三至四环', 'loc_ring_四至五环', 'loc_ring_二环内', 'loc_ring_内环内', 'loc_ring_内环至外环', 'loc_ring_外环外', 'loc_ring_内环至中环', 'loc_ring_中环至外环']
注意：以下环线值出现在 df 中但未列入 categories（示例）: ['二至三环']

示例（前8行）：
     环线  loc_ring_五至六环  loc_ring_六环外  loc_ring_三至四环  loc_ring_四至五环  \
0  二至三环            0.0           0.0            0.0            0.0   
1  五至六环            1.0           0.0            0.0            0.0   
2  五至六环            1.0           0.0            0.0            0.0   
3   六环外            0.0           1.0            0.0            0.0   
4  三至四环            0.0           0.0            1.0            0.0   
5  五至六环            1.0           0.0            0.0            0.0   
6   六环外            0.0           1.0            0.0            0.0   
7   六环外            0.0           1.0            0.0            0.0   

   loc_ring_二环内  loc_ring_内环内  loc_ring_内环至外环  loc_ring_外环外  loc_ring_内环至中环  \
0           0.0           0.0             0.0         

In [10]:
import pandas as pd

# 1. 检查空值（缺失值）数量
null_count = df_test['环线'].isnull().sum()
print(f"环线列的空值数量：{null_count}")

# 2. 检查唯一值（unique）数量
unique_count = df['环线'].nunique()
print(f"环线列的唯一值数量：{unique_count}")

# 3. 查看所有唯一值（中文描述会完整显示）
unique_values = df['环线'].unique()
print("环线列的所有唯一值：")
for value in unique_values:
    print(f"- {value}")

环线列的空值数量：0
环线列的唯一值数量：11
环线列的所有唯一值：
- 二至三环
- 五至六环
- 六环外
- 三至四环
- 四至五环
- 二环内
- 内环内
- 内环至外环
- 外环外
- 内环至中环
- 中环至外环


In [11]:
categories = [
   # "二至三环",-------预防虚拟变量陷阱
    "五至六环",
    "六环外",
    "三至四环",
    "四至五环",
    "二环内",
    "内环内",
    "内环至外环",
    "外环外",
    "内环至中环",
    "中环至外环"
]
cats_clean = [c.strip() for c in categories]

# 生成目标列名
dummy_cols = [f"loc_ring_{c}" for c in cats_clean]

# 初始化 dummiest DataFrame（float，以便放 NaN）
dummiest = pd.DataFrame(0.0, index=df_test.index, columns=dummy_cols)

# 原始列做 strip 处理以容忍前后空格
orig = df_test['环线'].astype(object).map(lambda x: x.strip() if pd.notna(x) else x)

# 对于原始为空的行，把所有虚拟列设为 NaN
mask_na = orig.isna()
dummiest.loc[mask_na, :] = np.nan

# 对非空行，按类别赋值（匹配 cats_clean）
mask_non_na = ~mask_na
for idx, val in orig[mask_non_na].items():
    if val in cats_clean:
        colname = f"loc_ring_{val}"
        dummiest.at[idx, colname] = 1.0
    else:
        # 如果遇到未列入 categories 的值，保持该行所有 dummy 为 0（并记录以便检查）
        pass

# 把这些新列拼回原 df_test
df_test = pd.concat([df_test, dummiest], axis=1)

# 打印检查信息
print("创建的虚拟变量列：", dummy_cols)
# 检查原始中未包含在 categories 内的值
orig_vals = set(df_test['环线'].dropna().astype(str).map(str.strip).unique())
missing_vals = sorted(list(orig_vals - set(cats_clean)))
if missing_vals:
    print("注意：以下环线值出现在 df_test 中但未列入 categories（示例）:", missing_vals[:30])
else:
    print("所有 df_test 中出现的非空环线值都已包含在 categories 中。")

# 示例查看
print("\n示例（前8行）：")
print(pd.concat([df_test['环线'].head(8), dummiest.head(8)], axis=1))


创建的虚拟变量列： ['loc_ring_五至六环', 'loc_ring_六环外', 'loc_ring_三至四环', 'loc_ring_四至五环', 'loc_ring_二环内', 'loc_ring_内环内', 'loc_ring_内环至外环', 'loc_ring_外环外', 'loc_ring_内环至中环', 'loc_ring_中环至外环']
注意：以下环线值出现在 df_test 中但未列入 categories（示例）: ['二至三环']

示例（前8行）：
     环线  loc_ring_五至六环  loc_ring_六环外  loc_ring_三至四环  loc_ring_四至五环  \
0  二至三环            0.0           0.0            0.0            0.0   
1  五至六环            1.0           0.0            0.0            0.0   
2  五至六环            1.0           0.0            0.0            0.0   
3   六环外            0.0           1.0            0.0            0.0   
4   二环内            0.0           0.0            0.0            0.0   
5   六环外            0.0           1.0            0.0            0.0   
6  四至五环            0.0           0.0            0.0            1.0   
7  四至五环            0.0           0.0            0.0            1.0   

   loc_ring_二环内  loc_ring_内环内  loc_ring_内环至外环  loc_ring_外环外  loc_ring_内环至中环  \
0           0.0           0.0             0.0    

#### 处理经纬度

In [12]:
import pandas as pd
import numpy as np
# 用 lon/lat 填补 coord_x/coord_y 的空值
print(f"coord_x 空值数（填补前）: {df['coord_x'].isna().sum()}")
print(f"coord_y 空值数（填补前）: {df['coord_y'].isna().sum()}")

df["coord_x"] = df["coord_x"].fillna(df["lon"])
df["coord_y"] = df["coord_y"].fillna(df["lat"])

print(f"coord_x 空值数（填补后）: {df['coord_x'].isna().sum()}")
print(f"coord_y 空值数（填补后）: {df['coord_y'].isna().sum()}")


coord_x 空值数（填补前）: 7131
coord_y 空值数（填补前）: 7131
coord_x 空值数（填补后）: 0
coord_y 空值数（填补后）: 0


In [13]:
import pandas as pd
import numpy as np
# 用 lon/lat 填补 coord_x/coord_y 的空值
print(f"coord_x 空值数（填补前）: {df_test['coord_x'].isna().sum()}")
print(f"coord_y 空值数（填补前）: {df_test['coord_y'].isna().sum()}")

df_test["coord_x"] = df_test["coord_x"].fillna(df_test["lon"])
df_test["coord_y"] = df_test["coord_y"].fillna(df_test["lat"])

print(f"coord_x 空值数（填补后）: {df_test['coord_x'].isna().sum()}")
print(f"coord_y 空值数（填补后）: {df_test['coord_y'].isna().sum()}")


coord_x 空值数（填补前）: 3715
coord_y 空值数（填补前）: 3715
coord_x 空值数（填补后）: 0
coord_y 空值数（填补后）: 0


In [14]:
# 保留4位小数
for col in ["lon", "lat", "coord_x", "coord_y"]:
    df[col] = df[col].round(4)
# 保留4位小数
for col in ["lon", "lat", "coord_x", "coord_y"]:
    df_test[col] = df_test[col].round(4)  

In [15]:
# 若四列仍不相等，则取平均重新赋值 lon/lat
mask_not_equal = (
    (df["lon"] != df["coord_x"]) | 
    (df["lat"] != df["coord_y"])
)

if mask_not_equal.any():
    df.loc[mask_not_equal, "lon"] = df.loc[mask_not_equal, ["lon", "coord_x"]].mean(axis=1)
    df.loc[mask_not_equal, "lat"] = df.loc[mask_not_equal, ["lat", "coord_y"]].mean(axis=1)
    print(f"⚠️ 已修正 {mask_not_equal.sum()} 行经纬度不一致的记录为平均值。")
else:
    print("✅ 保留三位小数后四列全部一致，无需平均。")

⚠️ 已修正 9255 行经纬度不一致的记录为平均值。


In [16]:
# 若四列仍不相等，则取平均重新赋值 lon/lat
mask_not_equal = (
    (df_test["lon"] != df_test["coord_x"]) | 
    (df_test["lat"] != df_test["coord_y"])
)

if mask_not_equal.any():
    df_test.loc[mask_not_equal, "lon"] = df_test.loc[mask_not_equal, ["lon", "coord_x"]].mean(axis=1)
    df_test.loc[mask_not_equal, "lat"] = df_test.loc[mask_not_equal, ["lat", "coord_y"]].mean(axis=1)
    print(f"⚠️ 已修正 {mask_not_equal.sum()} 行经纬度不一致的记录为平均值。")
else:
    print("✅ 保留三位小数后四列全部一致，无需平均。")

⚠️ 已修正 3731 行经纬度不一致的记录为平均值。


In [17]:
# 删除 coord_x, coord_y
df.drop(columns=["coord_x", "coord_y","物业办公电话"], inplace=True)
# 删除 coord_x, coord_y
df_test.drop(columns=["coord_x", "coord_y","物业办公电话"], inplace=True)

#### 建筑面积-套内面积比

In [18]:
import numpy as np

# 仅当两列都存在时才计算
if all(col in df.columns for col in ["建筑面积", "套内面积"]):
    df["建筑面积套内比"] = np.where(
        df["套内面积"].notna(),
        df["建筑面积"] / df["套内面积"],
        np.nan
    )
else:
    print("⚠️ 数据集中缺少 '建筑面积' 或 '套内面积' 列，无法计算比值。")
# 查看前几行
#df.head()


In [19]:
import numpy as np

# 仅当两列都存在时才计算
if all(col in df.columns for col in ["建筑面积", "套内面积"]):
    df_test["建筑面积套内比"] = np.where(
        df_test["套内面积"].notna(),
        df_test["建筑面积"] / df_test["套内面积"],
        np.nan
    )
else:
    print("⚠️ 数据集中缺少 '建筑面积' 或 '套内面积' 列，无法计算比值。")

# 查看前几行
#df_test.head()


### 虚拟变量+log(price)

In [20]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. 对Price取对数（处理偏态），生成因变量log_price
df['log_price'] = np.log(df['Price'])

# 2. 生成城市固定效应虚拟变量（避免共线性，删除第一个类别作为参照组）
city_dummies = pd.get_dummies(df['城市'], prefix='city', drop_first=True, dtype=int)
year_dummies = pd.get_dummies(df['年份'], prefix='year', drop_first=True, dtype=int)

# 2. 生成城市×年份交互固定效应虚拟变量（捕捉不同城市的不同年份趋势）
# 创建城市-年份交互项
df['city_year'] = df['城市'].astype(str) + '_' + df['年份'].astype(str)

# 生成交互固定效应虚拟变量（删除第一个作为参照组）
city_year_dummies = pd.get_dummies(df['city_year'], prefix='city_year', drop_first=True)

In [21]:
print("city_dummies列名:", city_dummies.columns.tolist()[:10])  # 显示前10个
print("year_dummies列名:", year_dummies.columns.tolist()[:60])

city_dummies列名: ['city_1', 'city_2', 'city_3', 'city_4', 'city_5', 'city_6', 'city_7', 'city_8', 'city_9', 'city_10']
year_dummies列名: ['year_2016.0', 'year_2017.0', 'year_2018.0', 'year_2019.0', 'year_2020.0', 'year_2021.0', 'year_2022.0']


In [22]:
loc1_dummies = pd.get_dummies(df['区域'], prefix='loc1', drop_first=True, dtype=int)
loc2_dummies = pd.get_dummies(df['板块'], prefix='loc2', drop_first=True, dtype=int)
loc3_dummies = pd.get_dummies(df['区县'], prefix='loc3', drop_first=True, dtype=int)
loc4_dummies = pd.get_dummies(df['板块_comm'], prefix='loc4', drop_first=True, dtype=int)

## KNN

##### 城市、区域、板块、Price、建筑面积、lon、lat、年份---训练集无缺失值的列

In [23]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix
from pandas.api.types import is_numeric_dtype

FEATURE_CATS = ["城市", "区域", "板块", "年份"] # 类别特征（虽为数值编码，本质是类别）
FEATURE_NUMS = ["lon", "lat", "建筑面积"]     # 数值特征（无缺失）
# 明确区分要当作类别处理的目标（即使它们是整数编码）!!!
categorical_targets = ["区县", "板块_comm"]
numeric_targets = ["建筑面积套内比"]
DESIRED_TARGETS = ["区县", "板块_comm", "建筑面积套内比"]
K = 8  # 近邻数量
EPS = 1e-6

n_rows = df.shape[0]# 获取数据总行数

# === 构建邻居特征空间（OneHot(CAT) + StandardScaler(NUM)） ===
# NOTE: FEATURE_CATS 已为数值编码，无需先转字符串；OneHotEncoder 能直接处理数值类别。
cat_matrix = df[FEATURE_CATS].astype(int).values  #转换为整数矩阵 shape (n_rows, n_cat)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_cat = ohe.fit_transform(cat_matrix)  # 独热编码后的稀疏矩阵（每行对应一个样本，列对应类别取值）

# 处理数值特征：标准化（Standardization）
num_matrix = df[FEATURE_NUMS].astype(float).values
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(num_matrix)
X_num_sparse = csr_matrix(X_num_scaled)

# 合并为稀疏矩阵
X_all = hstack([X_cat, X_num_sparse], format="csr")

In [24]:
# 加权众数函数（行向量版本）
def weighted_mode_for_row(vals, weights, allowed_set=None):
    vals = np.asarray(vals)
    w = np.asarray(weights)
    mask = ~pd.isnull(vals)
    if mask.sum() == 0:
        return np.nan
    vals = vals[mask]
    w = w[mask]
    agg = {}
    for v, wt in zip(vals, w):
        # 如果 allowed_set 存在且 v 不在 allowed_set，则跳过
        if (allowed_set is not None) and (v not in allowed_set):
            continue
        agg[v] = agg.get(v, 0.0) + float(wt)
    if len(agg) == 0:
        return np.nan
    # 返回权重最大的那个值
    return max(agg.items(), key=lambda x: x[1])[0]

In [25]:
# 如果存在全量 df（合并的 train+test），优先用它的已知类别作为 allowed_values，
# 否则用 df_test 中已知的类别
df_all_available = 'df' in globals() and isinstance(globals()['df'], pd.DataFrame)
allowed_values_map = {}
for cat in categorical_targets:
    if df_all_available and cat in globals()['df'].columns:
        allowed_values_map[cat] = set(globals()['df'][cat].dropna().unique())
    else:
        allowed_values_map[cat] = set(df_test[cat].dropna().unique())

# 对每个目标列逐一填补
for target in DESIRED_TARGETS:
    need_mask = df[target].isnull().to_numpy()
    n_missing = need_mask.sum()
    if n_missing == 0:
        print(f"[跳过] {target} 无缺失。")
        continue

    # 候选集：target值非缺失的行（这些行有已知值，可作为“邻居”的来源）
    cand_mask = ~pd.isnull(df[target]).to_numpy()
    cand_idx = np.where(cand_mask)[0]
    n_cand = len(cand_idx)
    if n_cand == 0:
        print(f"[跳过] {target} 在全表中无已知值，无法基于邻居填补。")
        continue

    # 拟合邻居搜索器（候选集）
    X_cand = X_all[cand_idx, :]
    nn = NearestNeighbors(n_neighbors=min(K, n_cand), metric="euclidean", n_jobs=-1)
    nn.fit(X_cand)

    # 待填索引与特征
    miss_pos = np.where(need_mask)[0]
    X_miss = X_all[miss_pos, :]

    k_use = min(K, n_cand)
    dist_mat, inds_mat = nn.kneighbors(X_miss, n_neighbors=k_use, return_distance=True)
    neigh_pos_mat = cand_idx[inds_mat]  # 映射回原索引
    weights = 1.0 / (dist_mat + EPS)

    if target in numeric_targets:
        # 数值目标：加权平均
        vals_mat = df[target].to_numpy(dtype=float)[neigh_pos_mat]  # shape (n_missing, k_use)
        mask_valid = ~np.isnan(vals_mat)
        numer = np.nansum(vals_mat * weights * mask_valid, axis=1)
        denom = np.sum(weights * mask_valid, axis=1)
        filled = np.where(denom > 0, numer / denom, np.nan)
        df.loc[df.index[miss_pos], target] = filled
    else:
        # 类别目标：加权众数，并确保结果属于 allowed_values_map[target]
        allowed_set = allowed_values_map.get(target, None)
        vals_all = df[target].to_numpy(dtype=object)  # 用于索引
        filled_list = []
        for r in range(neigh_pos_mat.shape[0]):
            neigh_pos = neigh_pos_mat[r]
            vals_row = vals_all[neigh_pos]
            w_row = weights[r]
            fm = weighted_mode_for_row(vals_row, w_row, allowed_set=allowed_set)
            # 如果得到的 fm 是 nan 或不在 allowed_set，再做后备：在邻居中取最常见且在 allowed_set 的值
            if pd.isnull(fm):
                # 后备：统计邻居中每个值的出现次数（非权重）
                uniq, counts = np.unique([v for v in vals_row if pd.notnull(v)], return_counts=True)
                if len(uniq) == 0:
                    # 最后备：从 allowed_set 里取任意一个（或设为 NaN）
                    fm = next(iter(allowed_set)) if (allowed_set and len(allowed_set)>0) else np.nan
                else:
                    # 按 count 选最近似值，但保证在 allowed_set
                    # 排序 uniq by counts desc
                    sorted_pairs = sorted(zip(uniq, counts), key=lambda x: -x[1])
                    chosen = None
                    for val, cnt in sorted_pairs:
                        if (allowed_set is None) or (val in allowed_set):
                            chosen = val
                            break
                    fm = chosen if chosen is not None else (next(iter(allowed_set)) if (allowed_set and len(allowed_set)>0) else np.nan)
            filled_list.append(fm)
        # 将填充值写回，若原列为整数，则尝试转成整数
        # 检查原始列 dtype
        if pd.api.types.is_integer_dtype(df[target].dtype) or all(pd.isnull(df[target]) | df[target].astype(str).str.isdigit()):
            # 先把 None/nan 保留为 NaN，再 cast int safely
            filled_arr = np.array([np.nan if pd.isnull(x) else int(x) for x in filled_list], dtype=float)
            df.loc[df.index[miss_pos], target] = filled_arr
            # 如果你想最终保留为 int dtype（无 NaN），可在确认无 NaN 后 cast to int
        else:
            df.loc[df.index[miss_pos], target] = filled_list

    print(f"[完成] {target} — 候选数={n_cand}, 待填数={n_missing}")

print("填补结束。现在 df 中这些列的唯一值（部分示例）：")
for t in DESIRED_TARGETS:
    print(t, "unique count:", df[t].nunique(), "sample values:", list(pd.Series(df[t].dropna().unique())[:10]))

[完成] 区县 — 候选数=96630, 待填数=7241
[完成] 板块_comm — 候选数=96293, 待填数=7578
[完成] 建筑面积套内比 — 候选数=35984, 待填数=67887
填补结束。现在 df 中这些列的唯一值（部分示例）：
区县 unique count: 109 sample values: [109.0, 65.0, 62.0, 123.0, 81.0, 112.0, 28.0, 68.0, 7.0, 5.0]
板块_comm unique count: 950 sample values: [150.0, 299.0, 911.0, 1102.0, 295.0, 78.0, 1126.0, 878.0, 547.0, 1123.0]
建筑面积套内比 unique count: 84032 sample values: [1.3355864337521413, 1.030234438156831, 1.157626287395782, 1.0, 1.330995323981296, 1.0386426098980253, 1.2398298816568047, 1.243513312096964, 1.139504328506593, 1.0566037735849054]


#### KNN测试集

In [26]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix
from pandas.api.types import is_numeric_dtype

FEATURE_CATS = ["城市", "区域", "板块", "年份"] # 类别特征（虽为数值编码，本质是类别）
FEATURE_NUMS = ["lon", "lat", "建筑面积"]     # 数值特征（无缺失）
# 明确区分要当作类别处理的目标（即使它们是整数编码）!!!
categorical_targets = ["区县", "板块_comm"]
numeric_targets = ["建筑面积套内比"]
DESIRED_TARGETS = ["区县", "板块_comm", "建筑面积套内比"]
K = 8  # 近邻数量
EPS = 1e-6

n_rows = df_test.shape[0]# 获取数据总行数

# === 构建邻居特征空间（OneHot(CAT) + StandardScaler(NUM)） ===
# NOTE: FEATURE_CATS 已为数值编码，无需先转字符串；OneHotEncoder 能直接处理数值类别。
cat_matrix = df_test[FEATURE_CATS].astype(int).values  #转换为整数矩阵 shape (n_rows, n_cat)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_cat = ohe.fit_transform(cat_matrix)  # 独热编码后的稀疏矩阵（每行对应一个样本，列对应类别取值）

# 处理数值特征：标准化（Standardization）
num_matrix = df_test[FEATURE_NUMS].astype(float).values
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(num_matrix)
X_num_sparse = csr_matrix(X_num_scaled)

# 合并为稀疏矩阵
X_all = hstack([X_cat, X_num_sparse], format="csr")

In [27]:
# 如果存在全量 df（合并的 train+test），优先用它的已知类别作为 allowed_values，
# 否则用 df_test 中已知的类别。
df_all_available = 'df' in globals() and isinstance(globals()['df'], pd.DataFrame)
allowed_values_map = {}
for cat in categorical_targets:
    if df_all_available and cat in globals()['df'].columns:
        allowed_values_map[cat] = set(globals()['df'][cat].dropna().unique())
    else:
        allowed_values_map[cat] = set(df_test[cat].dropna().unique())

# 对每个目标列逐一填补
for target in DESIRED_TARGETS:
    need_mask = df_test[target].isnull().to_numpy()
    n_missing = need_mask.sum()
    if n_missing == 0:
        print(f"[跳过] {target} 无缺失。")
        continue

    # 候选集（有该 target 的行）
    cand_mask = ~pd.isnull(df_test[target]).to_numpy()
    cand_idx = np.where(cand_mask)[0]
    n_cand = len(cand_idx)
    if n_cand == 0:
        print(f"[跳过] {target} 在全表中无已知值，无法基于邻居填补。")
        continue

    # 拟合邻居搜索器（候选集）
    X_cand = X_all[cand_idx, :]
    nn = NearestNeighbors(n_neighbors=min(K, n_cand), metric="euclidean", n_jobs=-1)
    nn.fit(X_cand)

    # 待填索引与特征
    miss_pos = np.where(need_mask)[0]
    X_miss = X_all[miss_pos, :]

    k_use = min(K, n_cand)
    dist_mat, inds_mat = nn.kneighbors(X_miss, n_neighbors=k_use, return_distance=True)
    neigh_pos_mat = cand_idx[inds_mat]  # 映射回原索引
    weights = 1.0 / (dist_mat + EPS)

    if target in numeric_targets:
        # 数值目标：加权平均
        vals_mat = df_test[target].to_numpy(dtype=float)[neigh_pos_mat]  # shape (n_missing, k_use)
        mask_valid = ~np.isnan(vals_mat)
        numer = np.nansum(vals_mat * weights * mask_valid, axis=1)
        denom = np.sum(weights * mask_valid, axis=1)
        filled = np.where(denom > 0, numer / denom, np.nan)
        df_test.loc[df_test.index[miss_pos], target] = filled
    else:
        # 类别目标：加权众数，并确保结果属于 allowed_values_map[target]
        allowed_set = allowed_values_map.get(target, None)
        vals_all = df_test[target].to_numpy(dtype=object)  # 用于索引
        filled_list = []
        for r in range(neigh_pos_mat.shape[0]):
            neigh_pos = neigh_pos_mat[r]
            vals_row = vals_all[neigh_pos]
            w_row = weights[r]
            fm = weighted_mode_for_row(vals_row, w_row, allowed_set=allowed_set)
            # 如果得到的 fm 是 nan 或不在 allowed_set，再做后备：在邻居中取最常见且在 allowed_set 的值
            if pd.isnull(fm):
                # 后备：统计邻居中每个值的出现次数（非权重）
                uniq, counts = np.unique([v for v in vals_row if pd.notnull(v)], return_counts=True)
                if len(uniq) == 0:
                    # 最后备：从 allowed_set 里取任意一个（或设为 NaN）
                    fm = next(iter(allowed_set)) if (allowed_set and len(allowed_set)>0) else np.nan
                else:
                    # 按 count 选最近似值，但保证在 allowed_set
                    # 排序 uniq by counts desc
                    sorted_pairs = sorted(zip(uniq, counts), key=lambda x: -x[1])
                    chosen = None
                    for val, cnt in sorted_pairs:
                        if (allowed_set is None) or (val in allowed_set):
                            chosen = val
                            break
                    fm = chosen if chosen is not None else (next(iter(allowed_set)) if (allowed_set and len(allowed_set)>0) else np.nan)
            filled_list.append(fm)
        # 将填充值写回，若原列为整数，则尝试转成整数
        # 检查原始列 dtype
        if pd.api.types.is_integer_dtype(df_test[target].dtype) or all(pd.isnull(df_test[target]) | df_test[target].astype(str).str.isdigit()):
            # 先把 None/nan 保留为 NaN，再 cast int safely
            filled_arr = np.array([np.nan if pd.isnull(x) else int(x) for x in filled_list], dtype=float)
            df_test.loc[df_test.index[miss_pos], target] = filled_arr
            # 如果你想最终保留为 int dtype（无 NaN），可在确认无 NaN 后 cast to int
        else:
            df_test.loc[df_test.index[miss_pos], target] = filled_list

    print(f"[完成] {target} — 候选数={n_cand}, 待填数={n_missing}")

print("填补结束。现在 df_test 中这些列的唯一值（部分示例）：")
for t in DESIRED_TARGETS:
    print(t, "unique count:", df_test[t].nunique(), "sample values:", list(pd.Series(df_test[t].dropna().unique())[:10]))

[完成] 区县 — 候选数=30285, 待填数=3732
[完成] 板块_comm — 候选数=30205, 待填数=3812
[完成] 建筑面积套内比 — 候选数=9915, 待填数=24102
填补结束。现在 df_test 中这些列的唯一值（部分示例）：
区县 unique count: 109 sample values: [109.0, 28.0, 123.0, 65.0, 60.0, 68.0, 95.0, 81.0, 55.0, 45.0]
板块_comm unique count: 914 sample values: [367.0, 606.0, 1110.0, 555.0, 990.0, 502.0, 575.0, 572.0, 173.0, 1139.0]
建筑面积套内比 unique count: 30154 sample values: [1.2233561736765541, 1.2318194483142937, 1.2549692549692548, 1.2421757266149844, 1.325399547627809, 1.1115603939865215, 1.2624464425804114, 1.1879849257370871, 1.2504456068785197, 1.2248937210223685]


#### 补全套内面积空值

In [28]:
df.loc[df['套内面积'].isna(), '套内面积'] = df['建筑面积'] / df['建筑面积套内比']
df_test.loc[df_test['套内面积'].isna(), '套内面积'] = df_test['建筑面积'] / df_test['建筑面积套内比']

#### 测试集加虚拟变量

loc1t_dummies = pd.get_dummies(df_test['区域'], prefix='loc1', drop_first=True, dtype=int)
loc2t_dummies = pd.get_dummies(df_test['板块'], prefix='loc2', drop_first=True, dtype=int)
loc3t_dummies = pd.get_dummies(df_test['区县'], prefix='loc3', drop_first=True, dtype=int)
loc4t_dummies = pd.get_dummies(df_test['板块_comm'], prefix='loc4', drop_first=True, dtype=int)

test_model= pd.concat([df_test[['建筑面积', '套内面积', '建筑面积套内比']],city_dummy,year_dummy,loc1t_dummies,loc2t_dummies,loc3t_dummies,loc4t_dummies], axis=1)


# 看列名（你会看到 city_1, city_2...）
print(city_dummy.columns.tolist()[:20])

#### 先给测试集加年份、城市的虚拟变量，因为其他的列有df中没有的数值

In [29]:
import pandas as pd

df_test['年份'] = df_test['年份'].apply(lambda x: 2022 if x == 2023 else x)

# 生成城市固定效应虚拟变量（避免共线性，删除第一个类别作为参照组）
city_dummy = pd.get_dummies(df_test['城市'], prefix='city', drop_first=True, dtype=int)

#### 由于测试集所有数据现在都是2022年了，就手动设置year_dummy

In [30]:
year_dummy = pd.DataFrame({'year_2022.0': np.ones(len(df_test), dtype=int)})

print("city_dummy 列:", city_dummy.columns.tolist())
print("year_dummy 列:", year_dummy.columns.tolist())
print("year_dummy 的值:", year_dummy['year_2022.0'].unique())

city_dummy 列: ['city_1', 'city_2', 'city_3', 'city_4', 'city_5', 'city_6', 'city_7', 'city_8', 'city_9', 'city_10', 'city_11']
year_dummy 列: ['year_2022.0']
year_dummy 的值: [1]


In [31]:
df_test['city_year'] = df_test['城市'].astype(str) + '_' + df_test['年份'].astype(str)
# 生成交互固定效应虚拟变量（删除第一个作为参照组）
#city_year_dummies = pd.get_dummies(df_test['city_year'], prefix='city_year', drop_first=True)
#print("city_year的值:", city_year_dummies.columns.tolist())

### 用来训练模型的df_model:

##### dummies代表环线虚拟变量

In [32]:
df_model= pd.concat([df[['建筑面积', '套内面积', '建筑面积套内比']],dummies,city_dummies,year_dummies,loc1_dummies,loc2_dummies,loc3_dummies,loc4_dummies], axis=1)

In [33]:
X = df_model
print(X)

          建筑面积        套内面积   建筑面积套内比  loc_ring_五至六环  loc_ring_六环外  \
0        52.30   39.158828  1.335586            0.0           0.0   
1       127.44  123.700000  1.030234            1.0           0.0   
2       118.02  101.950000  1.157626            1.0           0.0   
3       293.23  293.230000  1.000000            0.0           1.0   
4        39.85   29.940000  1.330995            0.0           0.0   
...        ...         ...       ...            ...           ...   
103866   93.60   76.490950  1.223674            0.0           1.0   
103867  132.00  113.805489  1.159874            0.0           1.0   
103868   92.00   74.757549  1.230645            0.0           1.0   
103869  123.00  103.542495  1.187918            0.0           1.0   
103870  144.00  121.551620  1.184682            0.0           1.0   

        loc_ring_三至四环  loc_ring_四至五环  loc_ring_二环内  loc_ring_内环内  \
0                 0.0            0.0           0.0           0.0   
1                 0.0            0.

### 开始OLS

In [34]:
y = df['log_price']

# 6. 添加常数项（OLS回归需要）
X = sm.add_constant(X)

# 7. 拟合OLS回归
model = sm.OLS(y, X).fit()
# 同样的操作比如制作虚拟变量,我需要

In [35]:
print(X.info())
print(X.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103871 entries, 0 to 103870
Columns: 2181 entries, const to loc4_1186.0
dtypes: float64(14), int32(2167)
memory usage: 869.7 MB
None
   const    建筑面积        套内面积   建筑面积套内比  loc_ring_五至六环  loc_ring_六环外  \
0    1.0   52.30   39.158828  1.335586            0.0           0.0   
1    1.0  127.44  123.700000  1.030234            1.0           0.0   
2    1.0  118.02  101.950000  1.157626            1.0           0.0   
3    1.0  293.23  293.230000  1.000000            0.0           1.0   
4    1.0   39.85   29.940000  1.330995            0.0           0.0   

   loc_ring_三至四环  loc_ring_四至五环  loc_ring_二环内  loc_ring_内环内  ...  loc4_1174.0  \
0            0.0            0.0           0.0           0.0  ...            0   
1            0.0            0.0           0.0           0.0  ...            0   
2            0.0            0.0           0.0           0.0  ...            0   
3            0.0            0.0           0.0           0.0  ...  

In [36]:
# 8. 输出回归结果
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.881
Method:                 Least Squares   F-statistic:                     650.7
Date:                Mon, 27 Oct 2025   Prob (F-statistic):               0.00
Time:                        22:21:20   Log-Likelihood:                -16992.
No. Observations:              103871   AIC:                         3.635e+04
Df Residuals:                  102689   BIC:                         4.764e+04
Df Model:                        1181                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             12.8507      0.273     46.

### 让训练集生成所有需要的虚拟变量

In [37]:

loc1t_dummies = pd.get_dummies(df_test['区域'], prefix='loc1', drop_first=True, dtype=float)
loc2t_dummies = pd.get_dummies(df_test['板块'], prefix='loc2', drop_first=True, dtype=float)
loc3t_dummies = pd.get_dummies(df_test['区县'], prefix='loc3', drop_first=True, dtype=float)
loc4t_dummies = pd.get_dummies(df_test['板块_comm'], prefix='loc4', drop_first=True, dtype=float)
# df_model= pd.concat([df[['建筑面积', '套内面积', '建筑面积套内比']],dummies,city_dummies,year_dummies,loc1_dummies,loc2_dummies,loc3_dummies,loc4_dummies], axis=1)
test_model= pd.concat([df_test[['ID','建筑面积', '套内面积', '建筑面积套内比']],dummiest,city_dummy,year_dummy,loc1t_dummies,loc2t_dummies,loc3t_dummies,loc4t_dummies], axis=1)

### 处理测试集相对df的特殊值

In [38]:
import pandas as pd
import numpy as np

def prepare_cleaned_test_and_extract_specials(X, test_model, id_col='ID', verbose=True):

    # 保证是 DataFrame
    X = pd.DataFrame(X)
    test_model = pd.DataFrame(test_model)

    # 训练集列（按此顺序对齐输出）
    X_cols = list(X.columns)

    # 1) 初始化 df_model_test_cleaned：先把 ID 放在最前（若 test_model 有 ID）
    out_cols = []
    if id_col in test_model.columns:
        out_cols.append(id_col)
    out_cols += X_cols

    # 创建输出 DataFrame，index 与 test_model 保持一致
    df_model_test_cleaned = pd.DataFrame(0.0, index=test_model.index, columns=out_cols, dtype=float)

    # 先把 ID 拷贝过去（如果存在）
    if id_col in test_model.columns:
        df_model_test_cleaned[id_col] = test_model[id_col].values

    # 2) 对 X 中的每一列：若 test_model 有则拷贝值，否则保持为 0
    for col in X_cols:
        if col in test_model.columns:
            # 强制 float，以便后续 OLS 可直接使用
            df_model_test_cleaned[col] = test_model[col].astype(float).values
        else:
            # 已经初始化为 0.0，无需额外操作（保留注释以示意）
            df_model_test_cleaned[col] = 0.0

    # 3) 识别 special_cols：test_model 中有但 X 中没有（去掉 ID）
    special_cols = [c for c in test_model.columns if (c not in X_cols) and (c != id_col)]
    # 按你的要求：这些 special_cols 不应出现在 df_model_test_cleaned 输出（我们没有把它们加入out_cols）

    # 4) 找出特殊元组：在 special_cols 上任一列非 0 的行（按原 test_model）
    if len(special_cols) == 0:
        # 无 special 列，返回空的 special_rows_df（方便 downstream）
        special_rows_df = pd.DataFrame(columns=[id_col] + [c for c in X_cols if c in test_model.columns])
        if verbose:
            print("No special columns (test-only dummies) detected. Nothing to extract.")
        # 返回已对齐的 cleaned 矩阵与空 special_rows_df
        return df_model_test_cleaned, special_rows_df

    # mask 表示那些在 special_cols 上有任意非零值的行
    # 使用 abs(sum) > 0 判定（支持浮点 fractional dummy）
    special_mask = (test_model[special_cols].abs().sum(axis=1) > 0)
    special_indices = test_model.index[special_mask]

    # 5) 构造 special_rows_df：包含 ID（若有）、所有 test 与 X 共有的列（common_cols）、以及 special_cols
    # common_cols 为 X_cols 与 test_model 列的交集（不含 ID）
    common_cols = [c for c in X_cols if c in test_model.columns]
    cols_for_special = []
    if id_col in test_model.columns:
        cols_for_special.append(id_col)
    cols_for_special += common_cols + special_cols

    special_rows_df = test_model.loc[special_indices, cols_for_special].copy()

    if verbose:
        print(f"训练列数 (X): {len(X_cols)}")
        print(f"测试列数 (test_model): {len(test_model.columns)}")
        print(f"测试缺少训练列数: {sum(1 for c in X_cols if c not in test_model.columns)}")
        print(f"测试多出训练外列数 (special cols): {len(special_cols)}")
        print("special_cols 示例（最多 30）:", special_cols[:30])
        print(f"被选作特殊元组的行数: {len(special_rows_df)}")

    # 注意：special_rows_df 保留了原 test_model 中的 ID 与行索引，便于后续按行将填充值写回 df_model_test_cleaned
    return df_model_test_cleaned, special_rows_df


In [39]:
df_cleaned, specials=prepare_cleaned_test_and_extract_specials(X, test_model, id_col='ID')

训练列数 (X): 2181
测试列数 (test_model): 2117
测试缺少训练列数: 103
测试多出训练外列数 (special cols): 38
special_cols 示例（最多 30）: ['loc1_3.0', 'loc1_61.0', 'loc1_79.0', 'loc2_10.0', 'loc2_77.0', 'loc2_110.0', 'loc2_144.0', 'loc2_278.0', 'loc2_358.0', 'loc2_499.0', 'loc2_507.0', 'loc2_559.0', 'loc2_613.0', 'loc2_634.0', 'loc2_659.0', 'loc2_691.0', 'loc2_699.0', 'loc2_706.0', 'loc2_726.0', 'loc2_848.0', 'loc2_859.0', 'loc2_1056.0', 'loc2_1099.0', 'loc2_1112.0', 'loc2_1127.0', 'loc2_1156.0', 'loc3_3.0', 'loc3_79.0', 'loc4_10.0', 'loc4_144.0']
被选作特殊元组的行数: 228


In [40]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

def fill_special_rows_by_hierarchical_knn(
    X,                          # 训练集模型矩阵（DataFrame），含 loc1_/loc2_/loc3_/loc4_ 列
    df_cleaned,                 # 已按 X.columns 初始化并对齐的测试矩阵（DataFrame），将被就地修改并返回
    special_rows_df,            # 包含 ID、common_cols、special_cols 的 DataFrame（index 与 df_cleaned 对齐）
    m=8,                        # 邻居个数
    loc_keys = ["loc1","loc2","loc3","loc4"],
    id_col = "ID",
    verbose = True
):
    """
    对 special_rows_df 中的每一特殊元组，按层级在 X 中筛候选并用 KNN 的邻居平均填充 df_cleaned 对应 loc 组列值。
    不使用 lon/lat/area；只用分类 dummy 列（common_cols）进行匹配与 KNN。
    返回修改后的 df_cleaned（修改为副本返回，输入不会被破坏）。
    """

    # 拷贝输出对象（避免原地修改导致意外）
    df_out = df_cleaned.copy()

    # 训练集的 loc 列分组（按前缀）
    X_cols = list(X.columns)
    #字典loc_cols_map，键是loc1，值是训练集中该前缀的所有列（如 loc1_10.0、loc1_20.0）
    loc_cols_map = {lk: [c for c in X_cols if c.startswith(lk + "_")] for lk in loc_keys}

    common_cols = [c for c in special_rows_df.columns if (c in X_cols) and (c != id_col)]

    special_cols = [c for c in special_rows_df.columns if (c not in X_cols) and (c != id_col)]

    if verbose:
        print("fill_special_rows_by_hierarchical_knn start")
        print("X cols:", len(X_cols), "loc groups:", {k:len(v) for k,v in loc_cols_map.items()})
        print("common_cols count:", len(common_cols), "special_cols count:", len(special_cols))
        print("special_rows count:", special_rows_df.shape[0])

    # 准备城市列集合（可能以 'city_' 开头）
    city_cols = [c for c in common_cols if c.startswith('city_')]
    # 高层 loc keys 顺序（城市>区域/区县>板块/板块_comm）
    # 我们用于放宽匹配时优先保留 loc1/loc3（更高层），然后 loc2/loc4（更细层）
    higher_loc = ['loc1','loc3']
    lower_loc = ['loc2','loc4']

    # 将 special_rows_df 的 common 列值转换为数值（float），用于比较/构造 prototype vectors
    # 保证索引与 df_out 对齐
    specials = special_rows_df.copy()


    # 遍历每一特殊元组（按行）
    for rid, row in specials.iterrows():
        proto_vals = {c: float(row[c]) if (c in common_cols) else None for c in specials.columns}
        # 找出该行哪些 loc 组在 special_cols 中有非零（即这些 loc 组是“多余的”/需要填补）
        missing_loc_keys = set()
        for spc in special_cols: # 遍历每一个特有列
            if pd.notna(row.get(spc)) and float(row.get(spc)) != 0.0:
                # special column like 'loc3_3.0' -> loc key 'loc3'
                for lk in loc_keys:
                    if spc.startswith(lk + "_"):
                        missing_loc_keys.add(lk) # 该loc组需要填充
                        break

        if len(missing_loc_keys) == 0:
            # 如果该行没有 special 非零项，跳过
            continue

        if verbose:
            print(f"\nProcessing row idx={rid}, ID={row.get(id_col,'-')}, missing locs={sorted(list(missing_loc_keys))}")

        # 剩下可用于匹配的 loc keys = 在 common_cols 中存在且该行对应有非零项的 loc keys
        known_loc_keys = []
        for lk in loc_keys:
            # 步骤1：找出当前loc组（如loc1）在common_cols中的所有列（如loc1_5.0、loc1_8.0）
            lk_cols_in_common = [c for c in loc_cols_map[lk] if c in common_cols]
            if not lk_cols_in_common:
                continue
          
          # 步骤2：检查当前行的这些列是否有非零值（有非零说明该loc组有已知值）
            vals = specials.loc[rid, lk_cols_in_common].astype(float)
            if (vals.abs().sum() > 0):
                # 步骤3：确保该loc组不在“需要填充的组”中（避免用待填充的组做匹配）
                if lk not in missing_loc_keys:
                    known_loc_keys.append(lk)

        def cols_for_lk_keys(lk_keys):
            cols = []
            # 步骤1：先加入城市列（城市优先级最高）
            cols += city_cols
            # 步骤2：加入输入loc组的列（如lk_keys=[loc1]，就加loc1在X_cols中的列）
            for lk in lk_keys:
                cols += [c for c in loc_cols_map[lk] if c in X_cols]
            return cols

        try_order = []
        if known_loc_keys:
        # 1) 先尝试用“所有已知loc组”匹配（最严格，相似性最高）
            try_order.append(tuple(sorted(known_loc_keys)))
        # 2) 再尝试用“高层loc组”匹配（若高层loc有值，优先用高层找）
        high_present = [lk for lk in higher_loc if lk in known_loc_keys]
        if high_present and tuple(high_present) not in try_order:
            try_order.append(tuple(sorted(high_present)))
        # 3) 再尝试用“loc1+loc2”匹配（兼顾高层和低层，补充可能的匹配）
        if 'loc1' in known_loc_keys and 'loc2' in known_loc_keys:
            t = tuple(sorted(['loc1','loc2']))
            if t not in try_order:
                try_order.append(t)
        # 4) 再尝试用“loc1单独”匹配（loc1是高层核心，单独作为匹配依据）
        if 'loc1' in known_loc_keys and ('loc1',) not in try_order:
            try_order.append(('loc1',))
        # 5) 最后尝试“仅用city列”匹配（最宽松，找不到其他候选时用）
        try_order.append(tuple())
#===============================================================================
        candidate_idx = []
        matched_on = None

        # 按try_order依次尝试匹配，直到找到有候选样本的层级
        for lk_tuple in try_order:
            # 步骤2：生成训练集的匹配掩码（mask=True的样本是候选）
            match_cols = cols_for_lk_keys(list(lk_tuple))
            if len(match_cols) == 0:
              
                mask = np.ones(len(X), dtype=bool)
            else:
                
                mask = np.ones(len(X), dtype=bool)
                for c in match_cols:
                    # 对每个匹配列，筛选“训练集列值≈当前行列值”的样本（容差1e-8，避免浮点误差）
                    pv = proto_vals.get(c, 0.0)
                    arr = X[c].fillna(0).astype(float).values
                    mask &= np.isclose(arr, float(pv), atol=1e-8)
            if mask.sum() > 0:
                candidate_idx = X.index[mask].tolist()
                matched_on = match_cols
                if verbose:
                    print(f"  Found {len(candidate_idx)} candidates by matching on cols: {match_cols}")
                break
            else:
                if verbose:
                    print(f"  No candidates when matching on cols: {match_cols}")

        # If still no candidates, fallback to entire X
        if len(candidate_idx) == 0:
            candidate_idx = X.index.tolist()
            if verbose:
                print("  Fallback: using entire training set as candidates (size=%d)" % len(candidate_idx))
#可以删除================================特征标准化！！！===========================
        # 用KNN从候选中找最相似的m个邻居（仅用matched_on列作为特征，不依赖经纬度
        if len(matched_on or []) == 0:
            neighbor_idx = candidate_idx[:m]
        else:
            feat_cols = matched_on.copy()

            cand_feat = X.loc[candidate_idx, feat_cols].fillna(0).astype(float)
            proto_feat = np.array([float(proto_vals.get(c, 0.0)) for c in feat_cols]).reshape(1, -1)

            scaler = StandardScaler()
            try:
                Xs = scaler.fit_transform(cand_feat.values)
                ps = scaler.transform(proto_feat)
            except Exception:

                Xs = cand_feat.values
                ps = proto_feat
            nn = NearestNeighbors(n_neighbors=min(m, Xs.shape[0])).fit(Xs)
            dists, idxs = nn.kneighbors(ps, return_distance=True)
            chosen_local = idxs[0].tolist()
            neighbor_idx = [cand_feat.index[i] for i in chosen_local]

        if verbose:
            print(f"  Selected {len(neighbor_idx)} neighbors (first 5): {neighbor_idx[:5]}")

        for missing_lk in missing_loc_keys:
            lk_cols = loc_cols_map.get(missing_lk, [])
            if len(lk_cols) == 0:
                if verbose:
                    print(f"   No train loc cols for {missing_lk}, skipping")
                continue
            neigh_vals = X.loc[neighbor_idx, lk_cols].astype(float)
            if neigh_vals.shape[0] == 0:

                mean_vec = X[lk_cols].astype(float).mean(axis=0).values
                if verbose:
                    print(f"   neighbors empty -> using global mean for {missing_lk}")
            else:
                mean_vec = neigh_vals.mean(axis=0).values

            for col_name, v in zip(lk_cols, mean_vec):
                df_out.at[rid, col_name] = float(v)

        if verbose:
            print(f"  Row {rid} filled for missing locs {sorted(list(missing_loc_keys))}")

    if verbose:
        print("fill_special_rows_by_hierarchical_knn finished.")
    return df_out


In [41]:
df_filled = fill_special_rows_by_hierarchical_knn(X, df_cleaned, specials, m=8, id_col='ID', verbose=True)

fill_special_rows_by_hierarchical_knn start
X cols: 2181 loc groups: {'loc1': 116, 'loc2': 976, 'loc3': 108, 'loc4': 949}
common_cols count: 2078 special_cols count: 38
special_rows count: 228

Processing row idx=7047, ID=1007047.0, missing locs=['loc2', 'loc4']
  Found 391 candidates by matching on cols: ['city_1', 'city_2', 'city_3', 'city_4', 'city_5', 'city_6', 'city_7', 'city_8', 'city_9', 'city_10', 'city_11', 'loc1_4.0', 'loc1_5.0', 'loc1_7.0', 'loc1_8.0', 'loc1_9.0', 'loc1_10.0', 'loc1_11.0', 'loc1_12.0', 'loc1_13.0', 'loc1_14.0', 'loc1_15.0', 'loc1_16.0', 'loc1_17.0', 'loc1_18.0', 'loc1_19.0', 'loc1_20.0', 'loc1_21.0', 'loc1_22.0', 'loc1_23.0', 'loc1_24.0', 'loc1_26.0', 'loc1_27.0', 'loc1_28.0', 'loc1_29.0', 'loc1_30.0', 'loc1_31.0', 'loc1_32.0', 'loc1_33.0', 'loc1_34.0', 'loc1_35.0', 'loc1_36.0', 'loc1_37.0', 'loc1_38.0', 'loc1_39.0', 'loc1_41.0', 'loc1_42.0', 'loc1_43.0', 'loc1_44.0', 'loc1_45.0', 'loc1_46.0', 'loc1_47.0', 'loc1_49.0', 'loc1_50.0', 'loc1_51.0', 'loc1_52.0', 

In [42]:
def print_dataframe_details(df, name="df_filled"):

    print("=" * 80)
    print(f"{name} 详细信息")
    print("=" * 80)
    
    # 1. 打印形状
    print(f"\n1. 形状 (行数, 列数): {df.shape}")
    print(f"   行数: {df.shape[0]:,}")
    print(f"   列数: {df.shape[1]:,}")
    
    # 2. 打印所有列名
    print(f"\n2. 所有列名 ({len(df.columns)} 列):")
    print("-" * 50)
    
    # 按字母顺序排序列名
    sorted_columns = sorted(df.columns.tolist())
    
    # 分组打印，每行打印5个列名
    for i in range(0, len(sorted_columns), 5):
        chunk = sorted_columns[i:i+5]
        print("   " + " | ".join(f"{col:<20}" for col in chunk))
    
    # 3. 按前缀统计列分布
    print(f"\n3. 列名前缀统计:")
    print("-" * 50)
    
    # 提取列名前缀（第一个下划线之前的部分）
    prefixes = {}
    for col in df.columns:
        if '_' in col:
            prefix = col.split('_')[0]
        else:
            prefix = col
        
        prefixes[prefix] = prefixes.get(prefix, 0) + 1
    
    # 按数量降序排列
    sorted_prefixes = sorted(prefixes.items(), key=lambda x: x[1], reverse=True)
    
    for prefix, count in sorted_prefixes:
        print(f"   {prefix}: {count}列")
    
    # 4. 打印数据基本信息
    print(f"\n4. 数据基本信息:")
    print("-" * 50)
    print(f"   内存使用: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    
    # 5. 打印前几行数据（如果列太多，只显示前几列）
    print(f"\n5. 数据预览 (前5行，前10列):")
    print("-" * 50)
    if df.shape[1] > 10:
        preview_cols = df.columns[:10].tolist()
        print(f"   (由于列数过多，只显示前10列: {preview_cols})")
        print(df[preview_cols].head())
    else:
        print(df.head())

# 使用示例
print_dataframe_details(df_filled, "df_filled")

df_filled 详细信息

1. 形状 (行数, 列数): (34017, 2182)
   行数: 34,017
   列数: 2,182

2. 所有列名 (2182 列):
--------------------------------------------------
   ID                   | city_1               | city_10              | city_11              | city_2              
   city_3               | city_4               | city_5               | city_6               | city_7              
   city_8               | city_9               | const                | loc1_10.0            | loc1_101.0          
   loc1_102.0           | loc1_103.0           | loc1_104.0           | loc1_105.0           | loc1_106.0          
   loc1_107.0           | loc1_108.0           | loc1_109.0           | loc1_11.0            | loc1_110.0          
   loc1_111.0           | loc1_112.0           | loc1_113.0           | loc1_114.0           | loc1_115.0          
   loc1_117.0           | loc1_118.0           | loc1_119.0           | loc1_12.0            | loc1_120.0          
   loc1_121.0           | loc1_122.0         

In [43]:
def merge_filled_into_cleaned(df_model_test_cleaned, df_filled, id_col='ID', inplace=False, verbose=True):
    """
    将 df_filled 中已填充的行按 ID 或行索引写回到 df_model_test_cleaned。
    - 仅写回两表共有的列（不包含 ID），按列名一一对应写回。
    - 保证 df_model_test_cleaned 的列顺序不变。
    - 返回新的 DataFrame（除非 inplace=True，则在原对象上修改并返回同一对象）。
    """
    if inplace:
        out = df_model_test_cleaned
    else:
        out = df_model_test_cleaned.copy()

    orig_cols = list(out.columns)

    if (id_col in out.columns) and (id_col in df_filled.columns):
        if verbose:
            print(f"[merge] aligning by ID column '{id_col}'")

        left = out.set_index(id_col, drop=False)
        right = df_filled.set_index(id_col, drop=False)

        common_ids = left.index.intersection(right.index)
        if len(common_ids) == 0:
            if verbose:
                print("[merge] warning: no matching IDs found between df_model_test_cleaned and df_filled.")

            return out.loc[:, orig_cols]

        cols_to_update = [c for c in left.columns if (c in right.columns) and (c != id_col)]
        if len(cols_to_update) == 0:
            if verbose:
                print("[merge] no common columns to update (excluding ID).")
            return out.loc[:, orig_cols]

        if verbose:
            print(f"[merge] will update {len(common_ids)} rows and {len(cols_to_update)} columns (sample cols: {cols_to_update[:10]})")

        left_subset = left.loc[common_ids, cols_to_update]
        right_subset = right.loc[common_ids, cols_to_update]

        left.loc[common_ids, cols_to_update] = right_subset

        out = left.reset_index(drop=True)

        out = out.loc[:, orig_cols]

        missing_ids = right.index.difference(left.index)
        if len(missing_ids) > 0 and verbose:
            print(f"[merge] {len(missing_ids)} IDs present in df_filled but not in cleaned (they were skipped). "
                  f"Sample: {list(missing_ids[:5])}")

        return out

    else:
        # fall back: align by index
        if verbose:
            print("[merge] ID column not present in both DataFrames; aligning by row index.")
        common_index = out.index.intersection(df_filled.index)
        if len(common_index) == 0:
            if verbose:
                print("[merge] warning: no overlapping row indices found; nothing to merge.")
            return out.loc[:, orig_cols]

        # columns to update = intersection of columns excluding ID if present
        cols_to_update = [c for c in out.columns if (c in df_filled.columns) and (c != id_col)]
        if len(cols_to_update) == 0:
            if verbose:
                print("[merge] no common columns to update (excluding ID).")
            return out.loc[:, orig_cols]

        if verbose:
            print(f"[merge] updating {len(common_index)} rows by index, columns: {cols_to_update[:10]}")

        # direct assignment for the overlapping index and columns
        out.loc[common_index, cols_to_update] = df_filled.loc[common_index, cols_to_update].values

        # ensure original column order
        out = out.loc[:, orig_cols]
        return out


In [44]:
# 假设 df_model_test_cleaned 已按 X.columns 初始化，df_filled 是 fill_special_rows_by_hierarchical_knn 的返回

df_model_test_cleaned = merge_filled_into_cleaned(df_cleaned, df_filled, id_col='ID', inplace=False, verbose=True)

# 快检
print("After merge: shape:", df_model_test_cleaned.shape)
# 检查 3 个示例 ID 是否更新（用你关心的列名）
#sample_ids = df_filled['ID'].dropna().unique()[:5].tolist()
#print("Sample IDs merged:", sample_ids)

[merge] aligning by ID column 'ID'
[merge] will update 34017 rows and 2181 columns (sample cols: ['const', '建筑面积', '套内面积', '建筑面积套内比', 'loc_ring_五至六环', 'loc_ring_六环外', 'loc_ring_三至四环', 'loc_ring_四至五环', 'loc_ring_二环内', 'loc_ring_内环内'])
After merge: shape: (34017, 2182)


### 检验测试集是否还有属性较多

In [45]:
# 1. 训练/测试列集合与差异检查（保持训练列的顺序）
train_cols = X.columns.tolist()
train_set = set(train_cols)
test_set = set(df_model_test_cleaned.columns.tolist())

missing_in_test = [c for c in train_cols if c not in test_set]   # 训练有、测试没有（保持训练顺序）
extra_in_test   = [c for c in df_model_test_cleaned.columns if c not in train_set]  # 测试有、训练没有

print("训练列总数:", len(train_cols))
print("测试列总数:", len(df_model_test_cleaned.columns))
print("测试缺少训练列数:", len(missing_in_test))
print("测试多出训练外列数:", len(extra_in_test))

if len(missing_in_test) > 0:
    print("测试缺少的前30列示例:", missing_in_test[:30])
if len(extra_in_test) > 0:
    print("测试多出的前30列示例:", extra_in_test[:30])

训练列总数: 2181
测试列总数: 2182
测试缺少训练列数: 0
测试多出训练外列数: 1
测试多出的前30列示例: ['ID']


### 测试集

In [46]:
df_model_test_cleaned['const'] = 1
#df_model_test_cleaned['lon'] = df_test['lon']
#df_model_test_cleaned['lat'] = df_test['lat']

print("const列已全部改为1")
print(f"const列的唯一值: {df_model_test_cleaned['const'].unique()}")

const列已全部改为1
const列的唯一值: [1]


In [47]:
df_model_test_cleaned = df_model_test_cleaned.drop('ID', axis=1)

In [48]:
# === 7. 使用训练好的 OLS 模型进行预测 ===
y_pred_log = model.predict(df_model_test_cleaned)

# === 8. 将预测结果反对数，得到原始价格 ===
y_pred = np.exp(y_pred_log)

In [49]:
print(y_pred.head())

0    4.035836e+07
1    2.621501e+06
2    4.578709e+06
3    2.800926e+06
4    1.059392e+07
dtype: float64


In [50]:
# 1. 先将预测价格添加到测试集df_test中（确保y_pred是你的预测结果）
df_test['预测价格'] = y_pred  # y_pred是模型输出的预测值，比如之前的y_pred变量

# 2. 同时选择ID和预测价格列，保存为CSV
df_test[['ID', '预测价格']].to_csv("ruc_Class25Q2_test_price_predicted.csv", index=False)

print("\n✅ 已完成预测，结果保存至 ruc_Class25Q2_test_price_predicted.csv")


✅ 已完成预测，结果保存至 ruc_Class25Q2_test_price_predicted.csv


In [51]:
import pandas as pd

# 读取测试文件
test_df = pd.read_csv("ruc_Class25Q2_test_rent.csv")

# 获取数据行数
n_rows = len(test_df)

# 计算后半段的起始索引（取后半部分）
start_index = 0

# 创建后半段的ID（保持原ID）
second_half_ids = test_df['ID'].iloc[start_index:]

# 创建后半段的price列，全部填充为0
second_half_prices = [0] * len(second_half_ids)

# 创建后半段的数据框
second_half_df = pd.DataFrame({
    'ID': second_half_ids,
    'price': second_half_prices
})

# 保存后半段数据
second_half_df.to_csv("test_rent_second_half.csv", index=False)

print(f"原文件总行数: {n_rows}")
print(f"后半段行数: {len(second_half_df)}")
print("后半段数据已保存为 'test_rent_second_half.csv'")

原文件总行数: 9773
后半段行数: 9773
后半段数据已保存为 'test_rent_second_half.csv'


In [52]:
import pandas as pd

# 读取两个文件
first_half = pd.read_csv("ruc_Class25Q2_test_price_predicted.csv")
second_half = pd.read_csv("test_rent_second_half.csv")

# 统一列名并选择需要的列
first_half = first_half.rename(columns={'预测价格': 'price'})[['ID', 'price']]
second_half = second_half[['ID', 'price']]

# 合并
final_df = pd.concat([first_half, second_half], ignore_index=True)

# 保存
final_df.to_csv("20251026submi.csv", index=False)

print("文件合并完成！")

文件合并完成！
